## PostProcess the Low Energy Transfer Data

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import numpy as np
import pylupnt as pnt
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# get moon position
et0 = 948341083.72  # where the moon is at 0 deg in the Sun-Earth rotating frame

n_sma = 7  # number of semi-major axis samples 4000 km - 16000 km (4000, 6000, 8000, 10000, 12000, 14000, 16000)
n_incs = 6  # number of inclination samples (40, 45, 50, 55, 60, 65)
n_Omega = 1  # number of RAAN samples
n_et = 16  # number of epoch samples
min_dv = 50  # m/s
max_dv = 1000  # m/s
d_dv = 1.0  # m/s

n_sat = n_sma * n_incs * n_Omega

datadir = "data/lowenergy_transfer/n_sma_{}_n_inc_{}_n_Omega_{}_dv_{}_{}_{:.1f}".format(
    n_sma, n_incs, n_Omega, int(min_dv), int(max_dv), d_dv
)

tspan = np.linspace(0, 3600 * 24 * 29.53, n_et, endpoint=False)  # 30 days
et0s = et0 + tspan

rows_keys = [
    "et0",
    "et_TLI",
    "tof_day",
    "dv_ms",
    "C3_mk2s2",
    "alt_TLI_km",
    "i_TLI_deg",
    "max_r_earth_km",
    "n_perigee",
    "n_perlilune",
    "coe_a_km",
    "coe_e",
    "coe_i_rad",
    "coe_Omega_rad",
    "coe_omega_rad",
    "coe_f_rad",
]

data = pd.DataFrame(columns=rows_keys)

for et0_sim in et0s:
    for si in range(n_sat):
        fname = datadir + f"/et_{et0_sim}" + f"/lowE_{et0_sim:.2f}_sat{si:02d}.csv"
        if os.path.exists(fname):
            # load csv
            data_load = pd.read_csv(fname)
            if not data_load.empty:
                data = pd.concat([data, data_load], ignore_index=True)
        else:
            print(f"File {fname} does not exist")

data

In [ ]:
import matplotlib.pyplot as plt
from src.low_energy_transfer import eci_to_serot

tspan = np.linspace(0, 3600 * 24 * 29.53, 3601)  # 30 days
et = et0 + tspan
moon_rv = pnt.get_body_pos_vel(et, pnt.EARTH, pnt.MOON, pnt.ECI)
moon_se = (
    eci_to_serot(et, moon_rv) * 1e-3
)  # moon position in the Sun-Earth rotating frame
theta = np.arctan2(moon_se[:, 1], moon_se[:, 0])
theta[theta < 0] += 2 * np.pi  # 0 to 2pi

# closest to 0, 90, 180, 270 deg
idxs = [
    np.argmin(np.abs(np.rad2deg(theta) - angle))
    for angle in np.linspace(0, 360, 16, endpoint=False)
]

fig = plt.figure(figsize=(6, 6))
ax = fig.add_subplot(111)
ax.plot(moon_se[:, 0], moon_se[:, 1], "k-")
for idx in idxs:
    ax.plot(moon_se[idx, 0], moon_se[idx, 1], "go")
    ax.text(
        moon_se[idx, 0],
        moon_se[idx, 1],
        f"{pnt.time_to_gregorian_string(et[idx], 0)}",
        fontsize=12,
    )
ax.plot(0, 0, "bo", label="Earth")
ax.set_xlabel("X [km]")
ax.set_ylabel("Y [km]")
ax.set_title("Moon orbit in the Sun-Earth rotating frame (30 days)")
ax.axis("equal")
ax.set_xlim(-4.5e5, 4.5e5)
ax.set_ylim(-4.5e5, 4.5e5)
ax.grid(True)
plt.tight_layout()
plt.savefig("figs/dv/moon_phase.pdf", dpi=300)
plt.show()

In [ ]:
smas = np.linspace(4000, 16000, n_sma)  # km
incs = np.deg2rad(np.array([40, 45, 50, 55, 60, 65]))
n_s_ = len(smas)
n_incs = len(incs)

n_params = 8  # et0, tof_day, dv_ms, coe
min_dv_params_all = (
    np.zeros((n_sma, len(incs), n_params)) + np.nan
)  # et0, tof_day, dv_ms
min_dv_params_direct = (
    np.zeros((n_sma, len(incs), n_params)) + np.nan
)  # et0, tof_day, dv_ms
min_dv_params_le = (
    np.zeros((n_sma, len(incs), n_params)) + np.nan
)  # et0, tof_day, dv_ms
min_dv_params_fb = (
    np.zeros((n_sma, len(incs), n_params)) + np.nan
)  # et0, tof_day, dv_ms


def store_params(row):
    coe = np.array(
        [
            row["coe_a_km"] * 1e3,  # km -> m
            row["coe_e"],
            row["coe_i_rad"],
            row["coe_Omega_rad"],
            row["coe_omega_rad"],
            row["coe_f_rad"],
        ]
    )
    params = np.zeros(n_params)
    params[0] = row["et0"]
    params[1] = row["tof_day"]
    params[2] = row["dv_ms"]
    params[3] = row["alt_TLI_km"]  # min altitude at TLI
    params[4] = row["C3_mk2s2"]  # C3 at TLI
    params[5] = row["max_r_earth_km"]  # max distance from Earth
    params[6] = row["n_perigee"]  # number of perigees
    params[7] = row["n_perlilune"]  # number of perlunes
    return params


for i, sma in enumerate(smas):
    for j, inc in enumerate(incs):
        df_sub = data[
            (np.abs(data["coe_a_km"] - sma) <= 1)
            & (np.abs(data["coe_i_rad"] - inc) <= np.deg2rad(0.5))
            & (data["alt_TLI_km"] < 1e4)
        ]  # filter by sma and inc and alt_TLI_km

        if not df_sub.empty:
            min_dv_row = df_sub.loc[df_sub["dv_ms"].idxmin()]
            min_dv_params_all[i, j, :] = store_params(min_dv_row)

            min_dv_direct = df_sub[(df_sub["tof_day"] < 7)]
            if not min_dv_direct.empty:
                min_dv_row_direct = min_dv_direct.loc[min_dv_direct["dv_ms"].idxmin()]
                min_dv_params_direct[i, j, :] = store_params(min_dv_row_direct)

            min_dv_fb = df_sub[(df_sub["tof_day"] >= 7) & (df_sub["C3_mk2s2"] < -2.0)]
            if not min_dv_fb.empty:
                min_dv_row_fb = min_dv_fb.loc[min_dv_fb["dv_ms"].idxmin()]
                min_dv_params_fb[i, j, :] = store_params(min_dv_row_fb)

            min_dv_le = df_sub[
                (df_sub["max_r_earth_km"] > 2.5 * 380000)
                & (df_sub["tof_day"] >= 30)
                & (df_sub["C3_mk2s2"] >= -1.0)
            ]
            if not min_dv_le.empty:
                min_dv_row_le = min_dv_le.loc[min_dv_le["dv_ms"].idxmin()]
                min_dv_params_le[i, j, :] = store_params(min_dv_row_le)

            print(
                f"SMA: {sma:.1f} km, INC: {np.rad2deg(inc):.1f} deg -> Min DV: {min_dv_row['dv_ms']:.1f} m/s at ET0: {min_dv_row['et0']}, TOF: {min_dv_row['tof_day']:.2f} days"
            )
        else:
            print(
                f"SMA: {sma:.1f} km, INC: {np.rad2deg(inc):.1f} deg -> No data available"
            )

# scatter plot of min_dv
# x: inc y: sma color: min_dv


def plot_min_dv(
    fig, ax, smas, incs, min_dv_params, title, add_colorbar=False, vmin=50, vmax=500
):

    sma_grid, inc_grid = np.meshgrid(smas, np.rad2deg(incs), indexing="ij")
    sc = ax.scatter(
        inc_grid,
        sma_grid,
        c=min_dv_params[:, :, 2],
        cmap="viridis",
        s=100,
        vmin=vmin,
        vmax=vmax,
    )

    # plot the feasible line for incliantion and eccentricity
    incvec = np.linspace(np.deg2rad(40), np.deg2rad(70), 100)
    evec = np.sqrt(1 - 5 / 3 * (np.cos(incvec)) ** 2)
    sma_feas = pnt.R_MOON / 1000 / (1 - evec)  # km
    ax.plot(
        np.rad2deg(incvec), sma_feas, "k--", label="Minimum Feasible SMA", linewidth=1
    )

    # add text labels
    for i in range(n_sma):
        for j in range(n_incs):
            min_dv_param = min_dv_params[i, j, :]
            min_dv = min_dv_param[2]
            C3 = min_dv_param[4]
            tof_day = min_dv_param[1]

            print_all = False

            if print_all:
                print_label = (
                    "{min_dv:.0f}m/s \n {tof_day:.1f}days \n {C3:.2f} km2/s2",
                )
            else:
                print_label = "{min_dv:.0f}m/s"

            if not np.isnan(min_dv):
                if print_all:
                    ax.text(
                        np.rad2deg(incs[j]) + 2,
                        smas[i] + 750,
                        print_label.format(min_dv=min_dv, tof_day=tof_day, C3=C3),
                        ha="center",
                        va="center",
                        color="black",
                        fontsize=7,
                    )
                else:
                    ax.text(
                        np.rad2deg(incs[j]) + 2,
                        smas[i] + 500,
                        print_label.format(min_dv=min_dv),
                        ha="center",
                        va="center",
                        color="black",
                        fontsize=12,
                    )
    ax.set_xlabel("Inclination (deg)", fontsize=12)
    ax.set_ylabel("Semi-Major Axis (km)", fontsize=12)
    ax.tick_params(axis="both", which="major", labelsize=10)  # major ticks
    ax.set_title(title, fontsize=16)
    ax.grid(True)
    ax.set_xticks(incs * 180 / np.pi)
    ax.set_yticks(smas)
    ax.set_ylim([3500, 17500])
    ax.set_xlim([35, 70])
    if add_colorbar:
        cbar = fig.colorbar(sc, ax=ax)
        cbar.set_label("Minimum Delta-V (m/s)")
    return sc


# Total
fig, ax = plt.subplots(1, 1, figsize=(8, 5))
plot_min_dv(
    fig,
    ax,
    smas,
    incs,
    min_dv_params_all,
    "Minimum Delta-V (All Transfers)",
    add_colorbar=True,
    vmin=50,
    vmax=800,
)
plt.savefig("figs/dv/min_dv_all_transfers.pdf", dpi=300)

fig, ax = plt.subplots(1, 1, figsize=(8, 5))
plot_min_dv(
    fig,
    ax,
    smas,
    incs,
    min_dv_params_fb,
    "Minimum Delta-V (Low C3 Transfers)",
    add_colorbar=True,
    vmin=50,
    vmax=800,
)
plt.savefig("figs/dv/min_dv_low_C3.pdf", dpi=300)

fig, ax = plt.subplots(1, 1, figsize=(8, 5))
plot_min_dv(
    fig,
    ax,
    smas,
    incs,
    min_dv_params_direct,
    "Minimum Delta-V (Direct Transfers)",
    add_colorbar=True,
    vmin=50,
    vmax=800,
)
plt.savefig("figs/dv/min_dv_direct_transfers.pdf", dpi=300)

fig, ax = plt.subplots(1, 1, figsize=(8, 5))
plot_min_dv(
    fig,
    ax,
    smas,
    incs,
    min_dv_params_le,
    "Minimum Delta-V (Low Energy Transfers)",
    add_colorbar=True,
    vmin=50,
    vmax=800,
)
plt.savefig("figs/dv/min_dv_low_energy_transfers.pdf", dpi=300)

In [ ]:
from src.low_energy_transfer import get_transfer_trajectory, eci_to_serot

# plots
sma_plots = [6000, 8000, 12000, 16000]  # plot trajectories for these sma only
inc_plots = [40]  # plot trajectories for this inclination only


def plot_min_dv(smas, incs, min_dv_params, title_str, bounds, figname, lw=1):
    fig, axes = plt.subplots(1, len(sma_plots), figsize=(4 * len(sma_plots), 4))

    # plot moon
    et0 = 948341083.72  # where the moon is at 0 deg in the Sun-Earth rotating frame
    tspan = np.linspace(0, 3600 * 24 * 30, 3601)  # 30 days
    et = et0 + tspan
    moon_rv = pnt.get_body_pos_vel(et, pnt.EARTH, pnt.MOON, pnt.ECI)
    moon_se = eci_to_serot(et, moon_rv)  # moon position in the Sun-Earth rotating frame

    idx = 0
    colors = ["blue", "green", "red", "orange", "purple"]

    for i, params in enumerate(min_dv_params):
        sma = smas[i]
        if sma not in sma_plots:
            continue

        for j, param in enumerate(params):
            inc = incs[j]
            if np.rad2deg(inc) not in inc_plots:
                continue

            if not np.isnan(param[2]):  # if dv_ms is not NaN
                et0 = param[0]
                tof_day = param[1]
                dv_mag = param[2]
                min_alt = param[3]
                C3 = param[4]
                max_r_earth = param[5]
                n_fbe = param[6]
                n_fbl = param[7]

                # reset sma, inc, Omega to the closest values
                coe_op = np.zeros(6)
                coe_op[0] = sma * 1e3  # sma
                coe_op[2] = inc  # inc
                coe_op[1] = np.sqrt(1 - 5 / 3 * np.cos(inc) ** 2)
                coe_op[3] = 0  # Omega
                coe_op[4] = np.deg2rad(90)  # omega
                coe_op[5] = 0  # f

                xprop_eci, xprop_mci, xprop_serot = get_transfer_trajectory(
                    et0, coe_op, dv_mag, prop_days=160
                )

                max_r_earth = (
                    np.max(np.linalg.norm(xprop_serot[:, 0:3], axis=1)) / 1e3
                )  # km
                max_idx = np.argmax(np.linalg.norm(xprop_serot[:, 0:3], axis=1))
                print(
                    f"Plotting trajectory for SMA: {smas[i]:.1f} km, INC: {np.rad2deg(incs[j]):.1f} deg"
                    + f"DV: {dv_mag:.1f} m/s, TOF: {tof_day:.2f} days, Min Alt: {min_alt:.1f} km  C3: {C3:.3f} km2/s2  Max RE: {max_r_earth:.2f} km"
                )

                # X-Y
                ax = axes[idx]
                ax.grid(True)
                ax.set_xlabel("X (km)", fontsize=14)
                ax.set_ylabel("Y (km)", fontsize=14)
                ax.set_title(
                    f"SMA: {smas[i]:.0f} km  Inc: {np.rad2deg(incs[j]):.1f} deg"
                    + f"\n Duration: {tof_day:.1f} days  DV: {dv_mag:.1f} m/s \n TLI Alt: {min_alt:.0f} km, C3: {C3:.3f} km2/s2",
                    fontsize=12,
                )
                ax.tick_params(axis="both", which="major", labelsize=12)  # major ticks
                ax.set_aspect("equal", "box")
                ax.plot(
                    xprop_serot[:, 0] / 1e3,
                    xprop_serot[:, 1] / 1e3,
                    color=colors[idx],
                    linewidth=lw,
                )
                ax.plot(
                    moon_se[:, 0] / 1e3,
                    moon_se[:, 1] / 1e3,
                    color="gray",
                    label="Moon Orbit",
                )
                # plot the first point
                ax.plot(
                    xprop_serot[0, 0] / 1e3,
                    xprop_serot[0, 1] / 1e3,
                    "o",
                    color=colors[idx],
                    linewidth=lw,
                )  # start point
                # plot the end point
                ax.plot(
                    xprop_serot[-1, 0] / 1e3,
                    xprop_serot[-1, 1] / 1e3,
                    "s",
                    color=colors[idx],
                    linewidth=lw,
                )  # end point
                # ax.legend(fontsize=8)
                ax.set_xlim([bounds[0], bounds[1]])
                ax.set_ylim([bounds[2], bounds[3]])

                idx += 1

    fig.tight_layout()
    if figname is not None:
        plt.savefig(figname, dpi=300)
    plt.show()


# First subplot: min_dv_params_all
plot_min_dv(
    smas,
    incs,
    min_dv_params_fb,
    "Transfer Trajectories (Direct Transfers with Flyby)",
    bounds=[-0.5e6, 0.5e6, -0.5e6, 0.5e6],
    figname="figs/dv/transfer_trajectories_low_C3.pdf",
)

# Second subplot: min_dv_params_50
plot_min_dv(
    smas,
    incs,
    min_dv_params_direct,
    "Transfer Trajectories (Direct Transfers, TOF < 7 days)",
    bounds=[-0.5e6, 0.5e6, -0.5e6, 0.5e6],
    figname="figs/dv/transfer_trajectories_direct.pdf",
)

# Third subplot: min_dv_params_le
plot_min_dv(
    smas,
    incs,
    min_dv_params_le,
    "Transfer Trajectories (Low Energy Transfers)",
    bounds=[-1.5e6, 1.5e6, -1.5e6, 1.5e6],
    figname="figs/dv/transfer_trajectories_low_energy.pdf",
)

In [ ]:
# plots
sma_plots = [6000, 8000, 12000, 16000]  # plot trajectories for these sma only
inc_plots = [40]  # plot trajectories for this inclination only

fig, axes = plt.subplots(2, 2, figsize=(10, 6))
axes = axes.flatten()

for i, sma in enumerate(sma_plots):
    for j, inc in enumerate(inc_plots):
        df_sub = data[
            (np.abs(data["coe_a_km"] - sma) <= 1)
            & (np.abs(data["coe_i_rad"] - np.deg2rad(inc)) <= np.deg2rad(0.5))
            & (data["alt_TLI_km"] < 1e4)
        ]  # filter by sma and inc and alt_TLI_km

        if not df_sub.empty:
            tof_day = df_sub["tof_day"]
            dv_mag = df_sub["dv_ms"]
            C3 = df_sub["C3_mk2s2"]

            # plot TOF vs DV with color C3
            ax = axes[i]
            sc = ax.scatter(
                tof_day, dv_mag, c=C3, cmap="viridis", s=20, vmin=-2.5, vmax=-0.5
            )

            # red circle for all direct transfers
            direct_transfers = df_sub[df_sub["tof_day"] < 7]

            cbar = fig.colorbar(sc, ax=ax, label="C3 (km2/s2)")
            ax.set_title(f"SMA: {sma:.0f} km, Inc: {inc:.1f} deg", fontsize=14)
            ax.set_xlabel("Duration (days)", fontsize=12)
            ax.set_ylabel("LOI Delta-V (m/s)", fontsize=12)
            ax.set_xlim([0, 170])
            ax.set_xticks([0, 20, 40, 60, 80, 100, 120, 140, 160])
            ax.set_ylim([0, 1100])
            ax.set_yticks([0, 200, 400, 600, 800, 1000])
            ax.grid(True)

            # # plot TOF vs C3 with color DV
            # ax = axes[1, i]
            # sc = ax.scatter(
            #     tof_day, C3, c=dv_mag, cmap="viridis", s=20, vmin=50, vmax=1000
            # )
            # cbar = fig.colorbar(sc, ax=ax, label="LOI Delta-V (m/s)")
            # ax.set_xlabel("Duration (days)", fontsize=12)
            # ax.set_ylabel("C3 (km2/s2)", fontsize=12)
            # ax.set_xlim([0, 170])
            # ax.set_xticks([0, 20, 40, 60, 80, 100, 120, 140, 160])
            # ax.set_ylim([-2.5, -0.5])
            # ax.set_yticks(
            #     [-2.6, -2.4, -2.2, -2.0, -1.8, -1.6, -1.4, -1.2, -1.0, -0.8, -0.6, -0.4]
            # )
            # ax.grid(True)

            # # plot DV vs C3 with color TOF
            # ax = axes[2, i]
            # sc = ax.scatter(
            #     dv_mag, C3, c=tof_day, cmap="viridis", s=20, vmin=0, vmax=160
            # )
            # cbar = fig.colorbar(sc, ax=ax, label="Duration (days)")
            # ax.set_xlabel("LOI Delta-V (m/s)", fontsize=12)
            # ax.set_ylabel("C3 (km2/s2)", fontsize=12)
            # ax.set_xlim([0, 1100])
            # ax.set_xticks([0, 200, 400, 600, 800, 1000])
            # ax.set_ylim([-2.5, -0.5])
            # ax.set_yticks(
            #     [-2.6, -2.4, -2.2, -2.0, -1.8, -1.6, -1.4, -1.2, -1.0, -0.8, -0.6, -0.4]
            # )
            # ax.grid(True)

plt.tight_layout()
plt.savefig("figs/dv/tof_vs_dv_all_transfers.pdf", dpi=300)